In [ ]:
# Devices shared by 3 or more customers in 7 days;
MATCH (c1:Customer)-[u1:USED_DEVICE]->(d:Device)

WITH d, datetime(u1.time) AS window_start

MATCH (c2:Customer)-[u2:USED_DEVICE]->(d)

WHERE datetime(u2.time) >= window_start
  AND datetime(u2.time) < window_start + duration({days: 7})

WITH d,
     window_start,
     collect(DISTINCT c2.customer_id) AS customers,
     count(u2) AS usage_count

WHERE size(customers) >= 3

RETURN d.device_id AS device_id,
       window_start,
       window_start + duration({days: 7}) AS window_end,
       size(customers) AS customer_count,
       customers,
       usage_count

ORDER BY customer_count DESC, usage_count DESC
LIMIT 20;

In [ ]:
#Generate your suspicious cluster in Neo4j Graph View
MATCH (c:Customer)-[u:USED_DEVICE]->(d:Device)
WHERE d.device_id = "D_0c947fb45bb50e72"
  AND datetime(u.time) >= datetime("2024-10-25T23:05:32.631Z")
  AND datetime(u.time) < datetime("2024-10-25T23:05:32.631Z")
      + duration({days: 7})

OPTIONAL MATCH (c)-[p:PAID]->(m:Merchant)
WHERE p.transaction_id = u.transaction_id

RETURN c, u, d, p, m;

What this query does:
1. Finds payments with fraud_label = 1.
2. Identifies the device used for that particular fraudulent transaction.
3. Finds other customers who used the same device.
4. Returns the fraud customer, related customers, device and merchant

In [ ]:
#customers within 2 hops of a fraud-labelled payment
MATCH (fraudCustomer:Customer)-[fraudPayment:PAID]->(merchant:Merchant)
WHERE fraudPayment.fraud_label = 1

MATCH (fraudCustomer)-[fraudUsage:USED_DEVICE]->(device:Device)
WHERE fraudUsage.transaction_id = fraudPayment.transaction_id

MATCH (relatedCustomer:Customer)-[relatedUsage:USED_DEVICE]->(device)

WHERE relatedCustomer <> fraudCustomer

RETURN DISTINCT
       fraudCustomer,
       fraudPayment,
       merchant,
       device,
       relatedCustomer,
       relatedUsage

LIMIT 200;

What this query does:
1. Finds devices shared by three or more customers within seven days.
2. Identifies payments made through those devices during the qualifying windows.
3. Matches USED_DEVICE and PAID relationships using the same transaction_id.
4. Groups transactions by merchant.
5. Ranks merchants by the number of distinct qualifying transactions.

In [ ]:
#Merchants most often paid from shared devices
// Step 1: Find devices shared by 3+ customers
// within any rolling 7-day window

MATCH (c1:Customer)-[u1:USED_DEVICE]->(d:Device)

WITH d, datetime(u1.time) AS windowStart

MATCH (c2:Customer)-[u2:USED_DEVICE]->(d)

WHERE datetime(u2.time) >= windowStart
  AND datetime(u2.time) < windowStart + duration({days: 7})

WITH d,
     windowStart,
     collect(DISTINCT c2.customer_id) AS customers

WHERE size(customers) >= 3

// Step 2: Remove duplicate qualifying windows
WITH DISTINCT d, windowStart

// Step 3: Find payments made using those shared devices
MATCH (c:Customer)-[u:USED_DEVICE]->(d)
MATCH (c)-[p:PAID]->(m:Merchant)

WHERE u.transaction_id = p.transaction_id
  AND datetime(u.time) >= windowStart
  AND datetime(u.time) < windowStart + duration({days: 7})

// Step 4: Rank merchants
RETURN
       m.merchant_id AS merchant_id,
       m.name AS merchant_name,
       count(DISTINCT p.transaction_id) AS transaction_count,
       count(DISTINCT d.device_id) AS shared_device_count,
       collect(DISTINCT d.device_id) AS shared_devices

ORDER BY transaction_count DESC
LIMIT 20;